# hCKSAAP_UbSite human: residual hybrid benchmark on Kaggle

This notebook trains the Short-Range Expert and Long-Context Expert on the verified-context subset of the published hCKSAAP_UbSite human training set, fits the residual fusion from protein-grouped out-of-fold predictions, refits both experts, and compares with the authors' dataset-specific checkpoint on matched published test sites. The subset is a documented post-feasibility amendment, not a reproduction of training on every published site.

Select **GPU T4 x2** and enable **Internet** in Kaggle settings. Add a Kaggle secret named `GITHUB_TOKEN` with read access to the private repository; `HF_TOKEN` is optional. Training uses five outer folds and 15 epochs per candidate without early stopping. Separate fold workers use the two T4s concurrently; the final refit uses one T4. The released dataset files come from the Git submodule, and the public checkpoint is downloaded and checksum-verified below. Published `xN` protein identifiers are resolved as candidate UniProt isoforms and retained only when their 49-residue windows match the source records.

Enable Kaggle notebook **Persistence → Files only** before training. Intermediate files are written to `/kaggle/working`; for a different account or notebook, attach a saved output dataset and set `RESUME_RUN_DIR` in the paths cell. Kaggle persistence is best effort, so save a version or export the run directory at milestones.

## Repository and credentials

In [1]:
from pathlib import Path
import os, subprocess, sys

from kaggle_secrets import UserSecretsClient

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
HF_TOKEN = secret('HF_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Add the read-only GITHUB_TOKEN secret to this Kaggle notebook.')

REPO_ROOT = Path('/kaggle/working/Ubiquitination-Sites-Prediction')
REPO_REF = None  # Set to the printed commit SHA when restoring a prior run.
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
git_env['GITHUB_TOKEN'] = GITHUB_TOKEN
askpass = Path('/kaggle/working/ubipred_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
askpass.chmod(0o700)
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules',
                        'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
                        str(REPO_ROOT)], check=True, env=git_env)
    elif REPO_REF is None:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
    if REPO_REF is not None:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', REPO_REF],
                       check=True, env=git_env)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
                   check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/kaggle/working/ubipred_huggingface_cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

Cloning into '/kaggle/working/Ubiquitination-Sites-Prediction'...
Submodule 'replication/MMUbiPred' (https://github.com/PakhrinLab/MMUbiPred.git) registered for path 'replication/MMUbiPred'
Cloning into '/kaggle/working/Ubiquitination-Sites-Prediction/replication/MMUbiPred'...


Submodule path 'replication/MMUbiPred': checked out 'fb45202d0a91c58e74d73a9f8e4ad43dff4e48e5'
2e8dca39ee5589ba2f2e51160fe7be7070243d7a


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## Dependencies and accelerator check

In [2]:
%pip install -q -r experiment/requirements-colab.txt h5py gdown
%pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json, torch

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2 before training.')
print('Visible GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def run_live(command, env=None):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

Visible GPUs: ['Tesla T4', 'Tesla T4']


## Paths, source preflight and released checkpoint

The released source files are already in the Git submodule. The checkpoint is taken from an attached Kaggle dataset if present, or downloaded from the authors' public Drive folder. Its SHA-256 is checked before use. The independent test records are not loaded during this audit.

In [4]:
import hashlib, shutil

CONFIG = Path('experiment/configs/benchmarks/hcksaap_residual_verified_v2.json')
BENCHMARK = 'hcksaap'
RUN_DIR = Path('/kaggle/working/mmubipred_benchmark_runs') / 'hcksaap_verified_v3' / 'run'
SOURCE_DIR = Path('/kaggle/working/mmubipred_benchmark_sources') / BENCHMARK
MODEL_FILE = SOURCE_DIR / 'DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5'
MODEL_DRIVE_ID = '1v6u6N6JpqJs8bi_3wLhp5gcbS7uxsAwD'
MODEL_SHA256 = 'fef6859092626f94e6a7a70e644c37f2f68bf93ec00f251c2003458dccb90526'

# To restore a previously saved Kaggle output, attach it as Input and set this path
# to the directory ending in /hcksaap_verified_v3/run. Leave None for a new or persisted session.
RESUME_RUN_DIR = None
if RESUME_RUN_DIR is not None:
    previous = Path(RESUME_RUN_DIR)
    if not (previous / 'protocol.json').is_file():
        raise FileNotFoundError(f'Prior run protocol missing: {previous}')
    previous_protocol = json.loads((previous / 'protocol.json').read_text())
    if previous_protocol.get('config') != json.loads(CONFIG.read_text()):
        raise ValueError('Attached run uses a different hCKSAAP protocol; only hcksaap_verified_v3 can resume here.')
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise RuntimeError(f'Current run directory is not empty: {RUN_DIR}')
    shutil.copytree(previous, RUN_DIR, dirs_exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()

if not MODEL_FILE.is_file():
    attached = list(Path('/kaggle/input').rglob(MODEL_FILE.name))
    if len(attached) > 1:
        raise RuntimeError(f'Multiple attached copies of the baseline: {attached}')
    if attached:
        shutil.copy2(attached[0], MODEL_FILE)
    else:
        import gdown
        gdown.download(id=MODEL_DRIVE_ID, output=str(MODEL_FILE), quiet=False)
if not MODEL_FILE.is_file() or sha256(MODEL_FILE) != MODEL_SHA256:
    raise RuntimeError(f'Baseline checkpoint missing or checksum mismatch: {MODEL_FILE}')
print('Verified checkpoint:', MODEL_FILE)
print('Run directory:', RUN_DIR)

def stage_command(name, *extra):
    return [sys.executable, '-u', 'experiment/scripts/run_dataset_benchmark.py', name,
            '--config', CONFIG, '--run-dir', RUN_DIR, '--source-dir', SOURCE_DIR,
            '--model-file', MODEL_FILE, *extra]

def stage(name, *extra):
    run_live(stage_command(name, *extra))

stage('audit')
audit = json.loads((RUN_DIR / 'source_audit.json').read_text())
display(audit)
assert audit['source_preflight_passed'], audit['blockers']

Downloading...
From: https://drive.google.com/uc?id=1v6u6N6JpqJs8bi_3wLhp5gcbS7uxsAwD
To: /kaggle/working/mmubipred_benchmark_sources/hcksaap/DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5
100%|██████████| 36.6M/36.6M [00:00<00:00, 121MB/s]


Verified checkpoint: /kaggle/working/mmubipred_benchmark_sources/hcksaap/DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5
Run directory: /kaggle/working/mmubipred_benchmark_runs/hcksaap_verified_v3/run
$ /usr/bin/python3 -u experiment/scripts/run_dataset_benchmark.py audit --config experiment/configs/benchmarks/hcksaap_residual_verified_v2.json --run-dir /kaggle/working/mmubipred_benchmark_runs/hcksaap_verified_v3/run --source-dir /kaggle/working/mmubipred_benchmark_sources/hcksaap --model-file /kaggle/working/mmubipred_benchmark_sources/hcksaap/DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5
{
  "benchmark": "hcksaap",
  "training_started": false,
  "test_predictions_generated": false,
  "files": {
    "train": [
      {
        "filename": "Ubiquitylation_81_window_training_positive.fasta",
        "available": true,
        "sha256": "eb21edd63598b8c88bf88d5ad10b3b2e98ccb966e6e6222966805c1346bc42d6"
      },
      {
        "filename": "Ubiquityl

{'baseline': {'available': True,
  'dataset_assignment_evidence': 'authors upstream README/notebook filename mapping',
  'expected_filename': 'DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5',
  'input_shapes': [[None, 49, 31], [None, 49, 21], [None, 49]],
  'sha256': 'fef6859092626f94e6a7a70e644c37f2f68bf93ec00f251c2003458dccb90526'},
 'benchmark': 'hcksaap',
 'blockers': [],
 'files': {'test': [{'available': True,
    'filename': 'Ubiquitylation_81_window_Testing_positive.fasta',
    'sha256': 'd9dc3102f586580f1c5733ad967bd58872d3053792c8082c564d0d9ecde3c9d1'},
   {'available': True,
    'filename': 'Ubiquitylation_81_window_Testing_negative.fasta',
    'sha256': 'c6c91e0df130ccf4a4c5d8d685617d9ab3a200b8821c0c9a3255181f87fc7f2a'}],
  'train': [{'available': True,
    'filename': 'Ubiquitylation_81_window_training_positive.fasta',
    'sha256': 'eb21edd63598b8c88bf88d5ad10b3b2e98ccb966e6e6222966805c1346bc42d6'},
   {'available': True,
    'filename': 'Ubiquitylation

## Prepare training context

The source adapter validates the published training windows against UniProt protein sequences. The original 95% coverage rule failed a training-only feasibility audit: 10,996/12,194 (90.18%) sites had verified context, with 5,523 negative and 5,473 positive sites. The amended training floor is 90%; unmatched sites are excluded without padding or guessed coordinates. This stage does not access the published test split.

In [5]:
run_live([sys.executable, '-m', 'unittest', 'discover', '-s', 'experiment/tests',
          '-p', 'test_dataset_benchmark.py'])
try:
    stage('prepare-train')
except subprocess.CalledProcessError:
    feasibility = RUN_DIR / 'train_feasibility.json'
    if feasibility.exists():
        print('Training feasibility report:', feasibility.read_text(), flush=True)
    raise
display(json.loads((RUN_DIR / 'train_preparation.json').read_text()))

$ /usr/bin/python3 -m unittest discover -s experiment/tests -p test_dataset_benchmark.py
..........train UniProt canonical: attempted 1/1 missing accessions
train UniProt isoforms: reviewed 1/1; validated 1
.{
  "preprocessing": {
    "source_files": []
  },
  "validation": {
    "total": 2,
    "retained": 2,
    "coverage": 1.0,
    "reasons": {
      "validated": 2
    },
    "support": {
      "1": 1,
      "0": 1
    }
  },
  "coordinate_offset": 1,
  "training_coordinate_match_counts": {
    "0": 0,
    "1": 2
  },
  "cache_sha256": "bcca9d34710f24beae7c5d033870c171c2ae25443cce7577c8d5211e5152a0b4",
  "test_labels_used_for_selection": false,
  "isoform_resolution": {
    "status_counts": {
      "validated_candidate": 1
    },
    "unresolved_accessions": []
  },
  "coverage_policy": {
    "minimum": 0.95,
    "passed": true,
    "matched_cohort_only": false
  },
  "cohort_sha256": "b173c394386b1b591fc841f9e9ddfbb289bab82d2891ea1f531a9f85d51f949f"
}
train UniProt isoforms: review

{'cache_sha256': 'eb0d11e76b0bd9f35d30e443557047f1e9c92272fd639c87bf3bde7577651dd5',
 'cohort_sha256': 'baff807917a05c8d3a60538d9ef786aec8c2088305c9758d8712e4ec1db2ad94',
 'coordinate_offset': 1,
 'coverage_policy': {'matched_cohort_only': True,
  'minimum': 0.9,
  'passed': True},
 'isoform_resolution': {'status_counts': {'not_found': 6,
   'published_window_mismatch': 1,
   'validated_candidate': 170},
  'unresolved_accessions': ['O15078x2',
   'O75592x2',
   'Q13148x2',
   'Q71SY5x3',
   'Q96PU5x8',
   'Q9P275x2',
   'Q9UBB6x3']},
 'preprocessing': {'exclusions': {'unsupported_residue': 5},
  'preprocessed_sites': 12194,
  'raw_sites': 12199,
  'raw_support': {'0': 6106, '1': 6093},
  'source_files': [{'filename': 'Ubiquitylation_81_window_training_positive.fasta',
    'sha256': 'eb21edd63598b8c88bf88d5ad10b3b2e98ccb966e6e6222966805c1346bc42d6'},
   {'filename': 'Ubiquitylation_81_window_training_negative.fasta',
    'sha256': 'a25fae9a545c33d7e9801d75b6d46acd34ed63b4be27533d602cbe7

## Five-fold development on two T4 GPUs

GPU 0 processes folds 0, 2 and 4; GPU 1 processes folds 1 and 3. Each worker writes to separate fold directories. Rerunning this cell reuses completed fold predictions and resumes incomplete candidates from the last completed epoch. When both workers finish, a final pass assembles all out-of-fold predictions and fits the residual fusion.

In [6]:
import threading, time

def run_parallel_development():
    assignments = [(0, [0, 2, 4]), (1, [1, 3])]
    processes, readers = [], []
    def stream_output(process, gpu):
        for line in process.stdout:
            print(f'[T4 {gpu}] {line}', end='', flush=True)
    try:
        for gpu, folds in assignments:
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = str(gpu)
            env['OMP_NUM_THREADS'] = '2'
            env['TOKENIZERS_PARALLELISM'] = 'false'
            command = stage_command('develop', *[part for fold in folds
                                                  for part in ('--fold', str(fold))])
            print(f'Starting GPU {gpu} on folds {folds}', flush=True)
            process = subprocess.Popen([str(part) for part in command], env=env,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, bufsize=1)
            processes.append((gpu, process))
            reader = threading.Thread(target=stream_output, args=(process, gpu), daemon=True)
            reader.start()
            readers.append(reader)
        while True:
            failed = [(gpu, p.returncode) for gpu, p in processes
                      if p.poll() is not None and p.returncode != 0]
            if failed:
                raise RuntimeError(f'Fold worker failed: {failed}')
            if all(p.poll() == 0 for _, p in processes):
                break
            time.sleep(1)
    except BaseException:
        for _, process in processes:
            if process.poll() is None:
                process.terminate()
        for _, process in processes:
            process.wait()
        raise
    finally:
        for reader in readers:
            reader.join(timeout=5)

run_parallel_development()
stage('develop')
display(json.loads((RUN_DIR / 'development_summary.json').read_text()))

Starting GPU 0 on folds [0, 2, 4]
Starting GPU 1 on folds [1, 3]
[T4 1] fold=1 expert=local seed=42 budget=15
[T4 0] fold=0 expert=local seed=42 budget=15
[T4 1] epoch=001 started
[T4 0] epoch=001 started
[T4 1] epoch=001 loss=0.90492 val_mcc@0.5=0.00000 val_auprc=0.60835 mean_branch_positive_probabilities=[0.9969916343688965, 0.7293834090232849, 0.9999999403953552]
[T4 0] epoch=001 loss=0.89654 val_mcc@0.5=0.03488 val_auprc=0.68805 mean_branch_positive_probabilities=[0.9938167333602905, 0.5684859156608582, 2.3694308310950873e-06]
[T4 1] epoch=002 started
[T4 0] epoch=002 started
[T4 1] epoch=002 loss=0.70960 val_mcc@0.5=0.24788 val_auprc=0.64197 mean_branch_positive_probabilities=[0.9989343285560608, 0.5279951691627502, 0.9999884963035583]
[T4 1] epoch=003 started
[T4 0] epoch=002 loss=0.70329 val_mcc@0.5=0.38532 val_auprc=0.73528 mean_branch_positive_probabilities=[0.9984824657440186, 0.4263627827167511, 5.458490864168608e-12]
[T4 0] epoch=003 started
[T4 1] epoch=003 loss=0.66736 va

{'benchmark': 'hcksaap',
 'comparison_valid': True,
 'contract': {'cohort_sha256': 'baff807917a05c8d3a60538d9ef786aec8c2088305c9758d8712e4ec1db2ad94',
  'fold_sha256': '29862cc1ff3bc378defa7bc8ce5c8a4d0021d3c791336b22b096809f229c2222',
  'preparation_sha256': '16c80918f737df82cc28665444f5e98f9f822cae0b4f3d7668cc3edc65ec5df2',
  'protocol_sha256': '653cd32dbf81933dfad729335dd9b60fbfec5317518e7711129327aadea5ce7f'},
 'crossfit_caveat': 'Fusion cross-fit over first-level OOF predictions; not fully nested end-to-end assessment',
 'fold_predictions_sha256': {'folds/fold_0/context/outer_predictions.npz': 'd4a223e731e737d8a16fee78756afa8b82ed2ca1ecbd4fee53d35a3395b5130c',
  'folds/fold_0/local/outer_predictions.npz': 'cd51c7b9d4c4a159bafd06df0575a6304870656fec8a451f2e0e36a03196f86a',
  'folds/fold_1/context/outer_predictions.npz': '1a3099c505a4f1dc305766907c07246d18bfe5bfd6dc5d2d2fea523f2dcd33fe',
  'folds/fold_1/local/outer_predictions.npz': 'bdb1e2326b23a178eb54271fc262702eedfb61a5e43994560

## Full training-set refit and model lock

Each expert is refitted once on the eligible published training data for the median best epoch selected in development. The fusion weights and decision threshold are then frozen.

In [7]:
stage('refit')
display(json.loads((RUN_DIR / 'model_lock.json').read_text()))

$ /usr/bin/python3 -u experiment/scripts/run_dataset_benchmark.py refit --config experiment/configs/benchmarks/hcksaap_residual_verified_v2.json --run-dir /kaggle/working/mmubipred_benchmark_runs/hcksaap_verified_v3/run --source-dir /kaggle/working/mmubipred_benchmark_sources/hcksaap --model-file /kaggle/working/mmubipred_benchmark_sources/hcksaap/DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5
full refit expert=local epochs=12 sites=10996
refit_epoch=001/012 started
refit_epoch=001/012 loss=0.84659
refit_epoch=002/012 started
refit_epoch=002/012 loss=0.67351
refit_epoch=003/012 started
refit_epoch=003/012 loss=0.64439
refit_epoch=004/012 started
refit_epoch=004/012 loss=0.62822
refit_epoch=005/012 started
refit_epoch=005/012 loss=0.62070
refit_epoch=006/012 started
refit_epoch=006/012 loss=0.61163
refit_epoch=007/012 started
refit_epoch=007/012 loss=0.60417
refit_epoch=008/012 started
refit_epoch=008/012 loss=0.59620
refit_epoch=009/012 started
refit_epoch=009/012 l

{'analysis_role': 'post-feasibility cohort-amended dataset-specific benchmark extension; not an architecture-only comparison on the full published training set',
 'baseline_sha256': 'fef6859092626f94e6a7a70e644c37f2f68bf93ec00f251c2003458dccb90526',
 'benchmark': 'hcksaap',
 'checkpoints': {'context': 'e624a9cf95aa1d9546afba0aae6e9ac1ab4c30c24d3bc80fc5791966e706d0ab',
  'local': 'c9b23410c59db922c14662f984781f24155cfb340b8dd7dcadd21f44c1e4ebba'},
 'development_summary_sha256': 'b125f1150541d2bb68fb38e2d395e0daa36d1a91d476ba08ae3c52761eb7f842',
 'protocol_sha256': '653cd32dbf81933dfad729335dd9b60fbfec5317518e7711129327aadea5ce7f',
 'refit_plan': {'context': {'epochs': 7, 'seed': 42},
  'local': {'epochs': 12, 'seed': 42}},
 'stacker': {'context_weight': 0.2357702612446995,
  'intercept': -0.02018472682474785,
  'l2_strength': 0.01,
  'local_weight': 0.7645275646644715},
 'test_inference_performed': False,
 'threshold': 0.5,
 'training_cohort_sha256': 'baff807917a05c8d3a60538d9ef786aec8c

## Published test preparation

Run only after the model lock exists. Both the released checkpoint and the newly trained hybrid will be evaluated on the same retained, context-valid test sites. Test coverage and exclusions are reported, but test labels or coverage are not used to revise the frozen model or threshold.

In [8]:
assert (RUN_DIR / 'model_lock.json').exists(), 'Complete the refit first.'
try:
    stage('prepare-test', '--allow-test')
except subprocess.CalledProcessError:
    feasibility = RUN_DIR / 'test_feasibility.json'
    if feasibility.exists():
        print('Test feasibility report:', feasibility.read_text(), flush=True)
    raise
display(json.loads((RUN_DIR / 'test_preparation.json').read_text()))

$ /usr/bin/python3 -u experiment/scripts/run_dataset_benchmark.py prepare-test --config experiment/configs/benchmarks/hcksaap_residual_verified_v2.json --run-dir /kaggle/working/mmubipred_benchmark_runs/hcksaap_verified_v3/run --source-dir /kaggle/working/mmubipred_benchmark_sources/hcksaap --model-file /kaggle/working/mmubipred_benchmark_sources/hcksaap/DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5 --allow-test
test UniProt canonical: attempted 40/1242 missing accessions
test UniProt canonical: attempted 80/1242 missing accessions
test UniProt canonical: attempted 120/1242 missing accessions
test UniProt canonical: attempted 160/1242 missing accessions
test UniProt canonical: attempted 200/1242 missing accessions
test UniProt canonical: attempted 240/1242 missing accessions
test UniProt canonical: attempted 280/1242 missing accessions
test UniProt canonical: attempted 320/1242 missing accessions
test UniProt canonical: attempted 360/1242 missing accessions
test Un

{'cache_sha256': '3fc4c4e07860e5bf348083fafce7902286484d1bbcfa4e901cb832737623d4e6',
 'cohort_sha256': '1a464cc3157edf94a2e133f456e25efdd9063ed2e03cf3690ba125a23e848f86',
 'coordinate_offset': 1,
 'coverage_policy': {'matched_cohort_only': True,
  'minimum': 0.0,
  'passed': True},
 'isoform_resolution': {'status_counts': {'published_window_mismatch': 2,
   'validated_candidate': 104},
  'unresolved_accessions': ['Q8TBF2x6', 'Q9NXE4x4']},
 'overlap': {'policy': 'published split preserved; overlap reported, not silently filtered',
  'test_sites_with_training_49mer': 30,
  'test_sites_with_training_protein': 0,
  'test_sites_with_training_site': 0},
 'preprocessing': {'exclusions': {'unsupported_residue': 5},
  'preprocessed_sites': 6804,
  'raw_sites': 6809,
  'raw_support': {'0': 3403, '1': 3406},
  'source_files': [{'filename': 'Ubiquitylation_81_window_Testing_positive.fasta',
    'sha256': 'd9dc3102f586580f1c5733ad967bd58872d3053792c8082c564d0d9ecde3c9d1'},
   {'filename': 'Ubiquity

## Legacy Keras comparator

The published HDF5 checkpoint uses legacy Keras layers. Install the matching `tf-keras` package without replacing Kaggle's existing TensorFlow dependencies.

In [9]:
import importlib.metadata as metadata

try:
    tf_version = metadata.version('tensorflow')
except metadata.PackageNotFoundError as error:
    raise RuntimeError('TensorFlow is absent from this Kaggle image; install a compatible TensorFlow CPU package before evaluation.') from error
legacy_requirement = 'tf-keras==' + '.'.join(tf_version.split('.')[:2]) + '.*'
run_live([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', legacy_requirement])

$ /usr/bin/python3 -m pip install -q --no-deps tf-keras==2.20.*


## Matched test comparison

The evaluation uses the locked hybrid, the authors' dataset-specific checkpoint, fixed threshold 0.5 and protein-cluster paired bootstrap. Keep the output together with the run directory.

In [10]:
stage('evaluate', '--allow-test')
result = json.loads((RUN_DIR / 'test_results.json').read_text())
display(result['metrics'])
display(result['paired_hybrid_vs_paper'])

$ /usr/bin/python3 -u experiment/scripts/run_dataset_benchmark.py evaluate --config experiment/configs/benchmarks/hcksaap_residual_verified_v2.json --run-dir /kaggle/working/mmubipred_benchmark_runs/hcksaap_verified_v3/run --source-dir /kaggle/working/mmubipred_benchmark_sources/hcksaap --model-file /kaggle/working/mmubipred_benchmark_sources/hcksaap/DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5 --allow-test

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 3690.33it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identic

{'context': {'accuracy': 0.6467637540453074,
  'auprc': 0.6507134903260219,
  'auroc': 0.6947138559343113,
  'confusion_matrix': [[1820, 1265], [918, 2177]],
  'f1': 0.6660547651828056,
  'mcc': 0.2952656229869375,
  'precision': 0.6324811156304474,
  'sensitivity': 0.7033925686591276,
  'specificity': 0.5899513776337115,
  'support': {'negative': 3085, 'positive': 3095, 'total': 6180},
  'threshold': 0.5},
 'hybrid': {'accuracy': 0.7176375404530745,
  'auprc': 0.756098540225207,
  'auroc': 0.7838881659392077,
  'confusion_matrix': [[1956, 1129], [616, 2479]],
  'f1': 0.7396688050126808,
  'mcc': 0.4412486350032895,
  'precision': 0.6870842572062085,
  'sensitivity': 0.8009693053311793,
  'specificity': 0.6340356564019449,
  'support': {'negative': 3085, 'positive': 3095, 'total': 6180},
  'threshold': 0.5},
 'local': {'accuracy': 0.7168284789644013,
  'auprc': 0.7575961244643719,
  'auroc': 0.7814250516465361,
  'confusion_matrix': [[1950, 1135], [615, 2480]],
  'f1': 0.73919523099850

{'confidence_level': 0.95,
 'method': 'paired nonparametric cluster bootstrap of candidate-minus-reference metric differences; whole canonical-accession protein groups resampled with replacement',
 'metrics': {'accuracy': {'candidate': 0.7176375404530745,
   'confidence_interval': [-0.05425079229077179, -0.032534824384886064],
   'difference': -0.04320388349514559,
   'paired_bootstrap_two_sided_p_value': 0.00019998000199980003,
   'reference': 0.7608414239482201},
  'auprc': {'candidate': 0.756098540225207,
   'confidence_interval': [-0.05786897126199529, -0.03148217701502947],
   'difference': -0.04456861550531077,
   'paired_bootstrap_two_sided_p_value': 0.00019998000199980003,
   'reference': 0.8006671557305177},
  'auroc': {'candidate': 0.7838881659392076,
   'confidence_interval': [-0.050232976294678655, -0.03296361971275842],
   'difference': -0.04164467706841424,
   'paired_bootstrap_two_sided_p_value': 0.00019998000199980003,
   'reference': 0.8255328430076219},
  'f1': {'cand

## Resumption and output transfer

With **Persistence → Files only**, rerun setup, paths and the interrupted stage after a session restart. Completed fold predictions are skipped; an interrupted epoch is repeated. For a different Kaggle account, save or export the entire `/kaggle/working/mmubipred_benchmark_runs/hcksaap_verified_v3/run` directory, attach it to the new notebook as Input, and set `RESUME_RUN_DIR` in the paths cell. Do not use the failed `hcksaap_isoform_v2/run` directory as a training run; its protocol differs. Use the same repository commit printed during the original run; changing the training implementation invalidates the recorded protocol. The public checkpoint can be downloaded again.

Kaggle's `/kaggle/working` files are not guaranteed to survive every failure. Save a notebook version with outputs or download a run snapshot before the session ends. The two-T4 parallelism applies to development folds; refit and final inference are single-process stages.